# 🧩 Mini-Lab: Boundary Schemas

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** system boundaries where untrusted data enters or leaves your application
2. **Define** dataclass schemas with type hints to describe the expected shape of data
3. **Validate** inputs and model outputs at boundaries so malformed data fails loudly and early

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Boundary Validation | Check user input and model output, the parts you do not control, so errors, abuse, and surprises are caught before they hit your core logic. |
| Dataclass Shape | Use a function for behavior and a dataclass for shape: a named, typed bundle of fields that makes pipeline inputs and outputs explicit. Use a class only when you truly need both state and behavior together. |
| Type Hints | Type hints are optional labels on variables and functions that describe what kinds of values you expect, helping tools catch bugs and making code easier to read and refactor. |


## Why Boundaries?

In an AI pipeline you typically control the code in the middle, but **not** the data at the edges:

| Boundary | Example of bad data |
|----------|---------------------|
| **User input** (HTTP request, form, CLI) | Missing field, wrong type, empty string |
| **Model output** (LLM JSON, API response) | Extra keys, null where you expect a list, hallucinated schema |

If you validate **at the boundary**, the rest of your code can trust the data it receives.

## Step 1 — Define a Schema with `@dataclass` and Type Hints

We'll model a simple extraction request: a user sends text and asks the LLM to pull out entities.

In [1]:
from dataclasses import dataclass, field


@dataclass
class ExtractionRequest:
    """Schema for an incoming extraction request."""
    text: str
    entity_types: list[str]
    max_entities: int = 10  # sensible default


# A well-formed request
good = ExtractionRequest(
    text="OpenAI was founded in 2015 in San Francisco.",
    entity_types=["person", "organization", "location"],
)
print(good)

ExtractionRequest(text='OpenAI was founded in 2015 in San Francisco.', entity_types=['person', 'organization', 'location'], max_entities=10)


The dataclass gives us a clear **shape**: we know exactly which fields exist and what types they should be. But a dataclass alone won't reject bad data — we need a validation step.

## Step 2 — Write a Boundary Validator

We write a small `validate_request` function that checks the raw dictionary (as it would arrive from an HTTP body or JSON file) and either returns a clean `ExtractionRequest` or raises a clear error.

In [2]:
class ValidationError(Exception):
    """Raised when input data fails boundary validation."""


def validate_request(raw: dict) -> ExtractionRequest:
    """Validate a raw dict at the system boundary and return a typed object."""
    errors: list[str] = []

    # --- Check required fields exist ---
    if "text" not in raw:
        errors.append("Missing required field: 'text'")
    elif not isinstance(raw["text"], str) or not raw["text"].strip():
        errors.append("'text' must be a non-empty string")

    if "entity_types" not in raw:
        errors.append("Missing required field: 'entity_types'")
    elif not isinstance(raw["entity_types"], list) or len(raw["entity_types"]) == 0:
        errors.append("'entity_types' must be a non-empty list")
    elif not all(isinstance(t, str) for t in raw["entity_types"]):
        errors.append("Every item in 'entity_types' must be a string")

    # --- Optional field with type check ---
    max_entities = raw.get("max_entities", 10)
    if not isinstance(max_entities, int) or max_entities < 1:
        errors.append("'max_entities' must be a positive integer")

    # --- Fail loudly with ALL errors at once ---
    if errors:
        raise ValidationError("Input validation failed:\n  • " + "\n  • ".join(errors))

    return ExtractionRequest(
        text=raw["text"].strip(),
        entity_types=raw["entity_types"],
        max_entities=max_entities,
    )

## Step 3 — Test with Good and Bad Payloads

Let's throw six payloads at the validator — three good, three bad — and see what happens.

In [3]:
test_payloads = [
    # ✅ Valid
    {"text": "Tesla reported $25B revenue.", "entity_types": ["organization", "money"]},
    # ✅ Valid with optional override
    {"text": "Alice met Bob in Paris.", "entity_types": ["person"], "max_entities": 5},
    # ❌ Missing 'text'
    {"entity_types": ["person"]},
    # ❌ Empty entity_types list
    {"text": "Some text here.", "entity_types": []},
    # ❌ Wrong type for text + bad max_entities
    {"text": 12345, "entity_types": ["org"], "max_entities": -1},
    # ❌ entity_types contains non-strings
    {"text": "Hello world.", "entity_types": ["person", 99]},
]

for i, payload in enumerate(test_payloads, 1):
    try:
        result = validate_request(payload)
        print(f"Payload {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Payload {i} ❌  → {exc}")
    print()

Payload 1 ✅  → ExtractionRequest(text='Tesla reported $25B revenue.', entity_types=['organization', 'money'], max_entities=10)

Payload 2 ✅  → ExtractionRequest(text='Alice met Bob in Paris.', entity_types=['person'], max_entities=5)

Payload 3 ❌  → Input validation failed:
  • Missing required field: 'text'

Payload 4 ❌  → Input validation failed:
  • 'entity_types' must be a non-empty list

Payload 5 ❌  → Input validation failed:
  • 'text' must be a non-empty string
  • 'max_entities' must be a positive integer

Payload 6 ❌  → Input validation failed:
  • Every item in 'entity_types' must be a string



Notice that **Payload 5** reports *two* errors at once — both the bad `text` type and the negative `max_entities`. Collecting all errors before raising gives the caller a single, complete error message instead of a frustrating fix-one-resubmit loop.

## Step 4 — Validate the Output Boundary

Model outputs are just as untrustworthy as user inputs. Let's define a schema for what we expect an LLM extraction to return, then validate simulated model output.

In [4]:
import json


@dataclass
class ExtractedEntity:
    """One entity extracted by the model."""
    name: str
    entity_type: str
    confidence: float


@dataclass
class ExtractionResponse:
    """Schema for the model's extraction output."""
    entities: list[ExtractedEntity]


def validate_response(raw_json: str) -> ExtractionResponse:
    """Validate model output at the output boundary."""
    # --- Parse JSON ---
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        raise ValidationError(f"Model returned invalid JSON: {exc}")

    if "entities" not in data or not isinstance(data["entities"], list):
        raise ValidationError("Model output missing 'entities' list")

    validated: list[ExtractedEntity] = []
    for idx, item in enumerate(data["entities"]):
        errors: list[str] = []
        if not isinstance(item.get("name"), str) or not item["name"].strip():
            errors.append(f"Entity [{idx}]: 'name' must be a non-empty string")
        if not isinstance(item.get("entity_type"), str):
            errors.append(f"Entity [{idx}]: 'entity_type' must be a string")
        conf = item.get("confidence")
        if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
            errors.append(f"Entity [{idx}]: 'confidence' must be a float between 0 and 1")
        if errors:
            raise ValidationError("Output validation failed:\n  • " + "\n  • ".join(errors))
        validated.append(ExtractedEntity(
            name=item["name"].strip(),
            entity_type=item["entity_type"],
            confidence=float(conf),
        ))

    return ExtractionResponse(entities=validated)

In [5]:
# Simulated model outputs — one good, two bad
model_outputs = [
    # ✅ Well-formed
    '{"entities": [{"name": "OpenAI", "entity_type": "organization", "confidence": 0.95}]}',
    # ❌ Invalid JSON (trailing comma)
    '{"entities": [{"name": "OpenAI",}]}',
    # ❌ confidence out of range
    '{"entities": [{"name": "Paris", "entity_type": "location", "confidence": 1.5}]}',
]

for i, output in enumerate(model_outputs, 1):
    try:
        result = validate_response(output)
        print(f"Output {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Output {i} ❌  → {exc}")
    print()

Output 1 ✅  → ExtractionResponse(entities=[ExtractedEntity(name='OpenAI', entity_type='organization', confidence=0.95)])

Output 2 ❌  → Model returned invalid JSON: Expecting property name enclosed in double quotes: line 1 column 33 (char 32)

Output 3 ❌  → Output validation failed:
  • Entity [0]: 'confidence' must be a float between 0 and 1



## 🎯 Summary

### Key Takeaways

1. **Boundary validation** — check data where it enters (user input) and where it leaves (model output) your system, so inner code can trust it
2. **Dataclass schemas** — `@dataclass` declares the exact shape (fields and defaults) you expect, serving as living documentation
3. **Type hints** — annotating fields with `str`, `int`, `list[str]`, `float` makes expectations explicit and enables runtime checks
4. **Collect all errors** — reporting every validation failure at once gives callers a single, actionable error message instead of a fix-one-resubmit loop
5. **Fail loudly** — a clear `ValidationError` at the boundary is far easier to debug than a mysterious `KeyError` deep inside your pipeline